<a href="https://colab.research.google.com/github/anushka-dev18/3Skill/blob/main/Fake_Job_Posting_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# STEP 1: Dependencies & Environment Setup
# =====================================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack

# Download NLTK text processing components
nltk.download('stopwords')
nltk.download('wordnet')

# Load the dataset (Ensure 'fake_job_postings.csv' is uploaded to Colab files)
df = pd.read_csv('fake_job_postings.csv')

print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nTarget Class Breakdown (0 = Legitimate, 1 = Fraudulent):")
print(df['fraudulent'].value_counts())
print("\nTarget Class Percentages:")
print(df['fraudulent'].value_counts(normalize=True) * 100)

# =====================================================================
# STEP 2: Exploratory Preprocessing & Text Cleaning
# =====================================================================
# Fill missing values for structured numerical features
df['has_company_logo'] = df['has_company_logo'].fillna(0)
df['has_questions'] = df['has_questions'].fillna(0)
df['telecommuting'] = df['telecommuting'].fillna(0)

# Fill missing values for text features
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in text_cols:
    df[col] = df[col].fillna('')

# Combine all textual features into a single master column
df['combined_text'] = (
    df['title'] + " " +
    df['company_profile'] + " " +
    df['description'] + " " +
    df['requirements'] + " " +
    df['benefits']
)

# Initialize text cleaning helpers
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove web links
    text = re.sub(r'[^a-zA-Z\s]', '', text)                                # Remove punctuation and numbers
    text = text.lower()                                                     # Convert to lowercase
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Remove stopwords & lemmatize
    return ' '.join(tokens)

print("\nCleaning combined text field (this takes ~1 minute)...")
df['clean_text'] = df['combined_text'].apply(clean_text)
print("Text cleaning complete.")

# =====================================================================
# STEP 3: Feature Extraction & Dataset Splitting
# =====================================================================
# Transform cleaned text into numerical vector space using TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_text = tfidf.fit_transform(df['clean_text'])

# Extract structured categorical/numerical features
X_num = df[['has_company_logo', 'has_questions', 'telecommuting']].values

# Combine TF-IDF sparse matrix with structured features
X_combined = hstack([X_text, X_num]).tocsr()
y = df['fraudulent'].values

# Stratified split to preserve class imbalance ratio across train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set count: {X_train.shape[0]}")
print(f"Testing set count:  {X_test.shape[0]}")

# =====================================================================
# STEP 4: Model Implementation & Evaluation
# =====================================================================
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

results = []

print("\nTraining and evaluating models...")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

# Display tabular metrics comparison
results_df = pd.DataFrame(results)
print("\n=== Model Performance Metrics ===")
display(results_df)

# =====================================================================
# STEP 5: Unseen Prediction System
# =====================================================================
best_model = models["Logistic Regression"]

def predict_job_posting(title, company_profile, description, requirements, benefits, has_logo, has_questions, telecommuting):
    raw_text = f"{title} {company_profile} {description} {requirements} {benefits}"
    cleaned = clean_text(raw_text)

    # Vectorize text and combine with structured inputs
    text_vec = tfidf.transform([cleaned])
    num_vec = np.array([[has_logo, has_questions, telecommuting]])
    input_vec = hstack([text_vec, num_vec]).tocsr()

    prediction = best_model.predict(input_vec)[0]
    probabilities = best_model.predict_proba(input_vec)[0]

    status = "⚠️ FRAUDULENT / FAKE" if prediction == 1 else "✅ LEGITIMATE / REAL"
    confidence = probabilities[prediction] * 100

    print("\n-------------------------------------------")
    print(f"Prediction: {status}")
    print(f"Confidence Level: {confidence:.2f}%")
    print(f"Estimated Fraud Probability: {probabilities[1]:.4f}")
    print("-------------------------------------------")

# Test Run 1: Suspicious / Scam-style Posting
print("\n--- TEST CASE 1: Suspicious Posting ---")
predict_job_posting(
    title="Work From Home Data Entry Clerk",
    company_profile="",
    description="Earn $5000 weekly typing simple documents online. Fast payouts. Must wire $100 for background check software.",
    requirements="Must have personal bank account.",
    benefits="Flexible hours, high weekly earnings.",
    has_logo=0,
    has_questions=0,
    telecommuting=1
)

# Test Run 2: Legitimate-style Posting
print("\n--- TEST CASE 2: Standard Tech Company Posting ---")
predict_job_posting(
    title="Senior Python / Backend Engineer",
    company_profile="DataTech Inc. is a leading enterprise cloud analytics platform founded in 2012...",
    description="We are seeking an experienced software engineer to build scalable REST APIs and backend ML pipelines.",
    requirements="4+ years experience with Python, FastAPI, PostgreSQL, and Docker containerization.",
    benefits="Health insurance, competitive salary, performance bonuses, 401(k) matching.",
    has_logo=1,
    has_questions=1,
    telecommuting=0
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Dataset Loaded Successfully: 17880 rows, 18 columns

Target Class Breakdown (0 = Legitimate, 1 = Fraudulent):
fraudulent
0    17014
1      866
Name: count, dtype: int64

Target Class Percentages:
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64

Cleaning combined text field (this takes ~1 minute)...
Text cleaning complete.

Training set count: 14304
Testing set count:  3576

Training and evaluating models...

=== Model Performance Metrics ===


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.9581,0.5401,0.8960,0.6739
1,Naive Bayes,0.9639,0.7245,0.4104,0.5240
2,Random Forest,0.9779,0.9796,0.5549,0.7085



--- TEST CASE 1: Suspicious Posting ---

-------------------------------------------
Prediction: ⚠️ FRAUDULENT / FAKE
Confidence Level: 97.50%
Estimated Fraud Probability: 0.9750
-------------------------------------------

--- TEST CASE 2: Standard Tech Company Posting ---

-------------------------------------------
Prediction: ✅ LEGITIMATE / REAL
Confidence Level: 98.68%
Estimated Fraud Probability: 0.0132
-------------------------------------------
